In [4]:
import pandas as pd
from haystack import Document
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.components.retrievers.in_memory import InMemoryBM25Retriever
from haystack.components.builders import PromptBuilder
from haystack_integrations.components.generators.ollama import OllamaGenerator
from haystack import Pipeline

/opt/anaconda3/lib/python3.12/site-packages/haystack/core/errors.py:34: DeprecationWarning: PipelineMaxLoops is deprecated and will be remove in version '2.7.0'; use PipelineMaxComponentRuns instead.
  warnings.warn(


In [5]:
dataset = pd.read_csv("/Users/archith25/Desktop/Data_PreProcessing/marine_ocr_output.csv")
docs1 = [
    {
        "content": f"{row['name']} - {row['significance']}. Coordinates: "
                   f"Latitudes: [{row['lat_1']}, {row['lat_2']}, {row['lat_3']}, {row['lat_4']}], "
                   f"Longitudes: [{row['lon_1']}, {row['lon_2']}, {row['lon_3']}, {row['lon_4']}]"
    }
    for index, row in dataset.iterrows()
]

docs = [Document(content=doc["content"]) for doc in docs1]

In [6]:
document_store = InMemoryDocumentStore()
document_store.write_documents(docs)

25

In [73]:
retriever_34 = InMemoryBM25Retriever(document_store)

In [74]:
template = """
Given the following context: 

{% for document in documents %}
    {{ document.content }}
{% endfor %}

You are a Maritime Situational Awareness system.
Parse through the input_text, and give Answer in the following format:

{
    Location: [<latitude, longitude>],
    Time: [<timestamp or time range>],
    Direction: [<north, east, west, south, etc.>],
    Object Type: [<ship, aircraft, submarine, etc.>],
    Object Description: [<size, color, movement, speed, direction, etc.>],
    Source: [<radar, satellite, human observation>],
    Confidence Level: [<high, medium, low>]
    Reference: "<Cross-reference the input_text with context>"
}

Input_Text: {{ input_text }}
Answer:
"""

template1 = """
Given the following context: 

{% for document in documents %}
    {{ document.content }}
{% endfor %}

Extract the following information from the input_text in the structured format:
(Give short answer)

{
    Location: [latitude, longitude],
    Time: [timestamp or time range],
    Direction: [north, east, west, south, etc.],
    Object Type: [ship, aircraft, submarine, etc.],
    Object Description: [size, color, movement, speed, direction, etc.],
    Source: [radar, satellite, human observation],
    Confidence Level: [high, medium, low]
    Refernce: 
}

If possible cross-reference the reports with information from context, if no cross-refernce return "no-cross reference".

You are a Maritime Situational Awareness model, give answers accordingly

Input_Text: {{ input_text }}
Answer:
"""

In [75]:
prompt_builder_34 = PromptBuilder(template=template)

generator_34 = OllamaGenerator(model = "gemma2:2b", 
                              url = "http://localhost:11434", 
                              generation_kwargs={"num_predict" : -2,
                                                 "temperature" : 0.7 
                                                }, 
                              timeout = 300
)

In [76]:
basic_rag_pipeline = Pipeline()
basic_rag_pipeline.add_component("retriever", retriever_34)
basic_rag_pipeline.add_component("prompt_builder", prompt_builder_34)
basic_rag_pipeline.add_component("llm", generator_34)
basic_rag_pipeline.connect("retriever", "prompt_builder.documents")
basic_rag_pipeline.connect("prompt_builder", "llm")

🚅 Components
  - retriever: InMemoryBM25Retriever
  - prompt_builder: PromptBuilder
  - llm: OllamaGenerator
🛤️ Connections
  - retriever.documents -> prompt_builder.documents (List[Document])
  - prompt_builder.prompt -> llm.prompt (str)

In [77]:
Input_text = """
A distress signal was received from a fishing vessel, "Oceanic Hope", located at approximately 12.345° N, 78.910° E. 
The vessel is experiencing engine failure and is drifting towards a reef. The vessel has a crew of 10 and is carrying 
a cargo of fish. Urgent assistance is required to prevent a potential maritime disaster.
"""

response1 = basic_rag_pipeline.run(
    {
        "retriever" : {"query": Input_text},
        "prompt_builder" : {"input_text": Input_text}
    }
)

print(response1["llm"]["replies"][0])

```json
{
    "Location": [12.345, 78.910],
    "Time": "Present",
    "Direction": "North-East",  
    "Object Type": "Fishing Vessel",
    "Object Description": "Engine failure, drifting towards a reef, crew of 10, cargo of fish.",
    "Source": "Distress Signal",
    "Confidence Level": "High",
    "Refernce": "A distress signal was received from the Oceanic Hope fishing vessel at coordinates 12.345° N, 78.910° E."
}
``` 


**Explanation:**

* **Location:** The provided latitude and longitude are used to determine the location of the distress signal.
* **Time:** "Present" is used since this response is based on real-time event information.  
* **Direction:** We're assuming direction from the perspective of where the vessel is drifting, so North-East is our best guess given the input text 
* **Object Type:** Based on distress signal and description, we assume it's a fishing vessel (Oceanic Hope)
* **Object Description:** The text gives us details about the vessel's condition and carg

In [85]:
import json
import re
"""
def extract_json(output):
    json_str = re.search(r'\{.*\}', output, re.DOTALL)
    if json_str:
        return json.loads(json_str.group())
    else:
        return None
"""
def extract_json(output):
    json_str = re.search(r'\{.*\}', output, re.DOTALL)
    if json_str:
        json_text = json_str.group()
        return json.loads(json_text)
    else:
        return None

json_data1 = extract_json(response1["llm"]["replies"][0])

historical_file = "/Users/archith25/Desktop/Data_PreProcessing/historical_data.json"

try:
    with open(historical_file, 'r') as f:
        try:
            historical_data = json.load(f)
            if not isinstance(historical_data, list):
                historical_data = []
        except json.JSONDecodeError:
            historical_data = []
except FileNotFoundError:
    historical_data = []

historical_data.extend([json_data1])

with open(historical_file, 'w') as f:
    json.dump(historical_data, f, indent=4)

print("Data saved successfully.")

Data saved successfully.


<>:3: SyntaxWarning: invalid escape sequence '\{'
<>:3: SyntaxWarning: invalid escape sequence '\{'
/var/folders/x2/p857nctj7hndgfwnttbcz4jh0000gn/T/ipykernel_906/2436455092.py:3: SyntaxWarning: invalid escape sequence '\{'
  """
